In [1]:
# 파일 상단 쪽, import 아래나 전처리 위에 추가
SPARSE_TH = 0.003   # 기존 0.005에서 살짝 완화
LOW_VAR_TH = 1e-6   # 그대로 두거나, 살짝 완화하고 싶으면 5e-7 정도

In [2]:
# 희소(sparse) 피처 제거 기준
#  - 각 컬럼에서 "0이 아닌 값의 비율"이 SPARSE_TH 미만이면 거의 안 쓰이는 변수로 보고 제거
#  - 기존 0.005 → 0.003 으로 완화:
#      · 너무 공격적으로 지우면, 드물게 등장하지만 성별 구분에 도움이 되는
#        브랜드/코너/바이어 관련 패턴까지 같이 사라지는 문제가 있어서
#      · 약간 더 많은 희소 피처를 살려 두고, CatBoost가 자체적으로 중요도 판단하도록 유도
# SPARSE_TH = 0.003   # 기존 0.005에서 살짝 완화

# Low-variance(낮은 분산) 피처 제거 기준
#  - 분산이 LOW_VAR_TH 미만이면 "거의 상수"로 간주하고 제거
#  - 트리 기반 모델(CatBoost)은 분산이 조금만 있어도 분할에 활용할 수 있기 때문에
#    기준을 매우 작게(1e-6) 두어, 진짜 상수 수준의 컬럼만 정리하고
#    유의미한 one-hot / 비율 피처들은 최대한 남기도록 설정
#  - 더 보수적으로 가고 싶으면 1e-6 → 5e-7 정도로 낮춰도 무방 (실험 결과에 따라 조정)
# LOW_VAR_TH = 1e-6   # 거의 상수인 피처만 제거하는 보수적인 기준

In [3]:
import pandas as pd
import numpy as np
import re
from scipy.stats import entropy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# 1. 데이터 로드 (인코딩 자동 시도)
# ============================================================
def robust_read_csv(path):
    for enc in ['cp949', 'euc-kr', 'utf-8']:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path)

train_tx = robust_read_csv('train_transactions.csv')
test_tx = robust_read_csv('test_transactions.csv')
y_train = robust_read_csv('y_train.csv')

train_tx['dataset'] = 'train'
test_tx['dataset'] = 'test'
full_tx = pd.concat([train_tx, test_tx], ignore_index=True)

# 누락 컬럼 방어적 생성
default_num_cols = ['tot_amt', 'dis_amt', 'net_amt', 'inst_mon', 'sales_time']
default_str_cols = ['brd_nm', 'corner_nm', 'part_nm', 'buyer_nm', 'str_nm']

for c in default_num_cols:
    if c not in full_tx.columns:
        full_tx[c] = 0
for c in default_str_cols:
    if c not in full_tx.columns:
        full_tx[c] = ''

# ============================================================
# 2. 쇼핑 스타일 피처 (명품 / 행사 / 할인 변동성)
# ============================================================
print("1) 쇼핑 스타일 피처 생성 중...")

def build_style_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data['corner_nm'] = data['corner_nm'].fillna('').astype(str)

    # 명품/수입 성향
    luxury_kw = ['수입', '명품', '부띠끄', '부틱', '디자이너']
    data['st_luxury_flag'] = data['corner_nm'].apply(
        lambda x: int(any(k in x for k in luxury_kw))
    )

    # 행사/알뜰 성향
    bargain_kw = ['행사', '균일가', '대전', '기획', '특가', '이월']
    data['st_bargain_flag'] = data['corner_nm'].apply(
        lambda x: int(any(k in x for k in bargain_kw))
    )

    # 할인율
    data['st_disc_ratio'] = data['dis_amt'] / (data['tot_amt'] + 1e-5)
    disc_std = data.groupby('custid')['st_disc_ratio'].std().fillna(0)
    disc_mean = data.groupby('custid')['st_disc_ratio'].mean().fillna(0)

    agg = data.groupby('custid')[['st_luxury_flag', 'st_bargain_flag']].mean()
    agg.columns = ['st_ratio_luxury', 'st_ratio_bargain']
    agg['st_disc_std'] = disc_std
    agg['st_disc_mean'] = disc_mean

    return agg

style_feats = build_style_features(full_tx)

# ============================================================
# 3. 행동 패턴 (환불 / 할부 / 다양성 / 고가 구매)
# ============================================================
print("2) 행동 패턴 피처 생성 중...")

def build_behavior_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    # 환불 여부 (net_amt 우선, 없으면 tot_amt)
    base_amt = np.where(
        (data['net_amt'].fillna(0) != 0),
        data['net_amt'].fillna(0),
        data['tot_amt'].fillna(0)
    )
    data['bh_refund_flag'] = (base_amt < 0).astype(int)

    refund_grp = data.groupby('custid')['bh_refund_flag'].agg(['sum', 'mean'])
    refund_grp.columns = ['bh_refund_cnt', 'bh_refund_rate']

    # 할부 사용
    data['bh_inst_flag'] = (data['inst_mon'] > 0).astype(int)
    inst_rate = data.groupby('custid')['bh_inst_flag'].mean().to_frame('bh_inst_rate')
    inst_max = data.groupby('custid')['inst_mon'].max().to_frame('bh_inst_max')

    # 브랜드/코너/파트 다양성
    div = data.groupby('custid')[['brd_nm', 'corner_nm', 'part_nm']].nunique()
    div.columns = ['bh_nunique_brd', 'bh_nunique_corner', 'bh_nunique_part']

    # 객단가 / 고가 구매 횟수
    amt_agg = data.groupby('custid')['tot_amt'].agg(['sum', 'count'])
    ticket = (amt_agg['sum'] / (amt_agg['count'] + 1e-5)).to_frame('bh_ticket_size')

    data['bh_high_amt_flag'] = (data['tot_amt'] > 300000).astype(int)
    hi_cnt = data.groupby('custid')['bh_high_amt_flag'].sum().to_frame('bh_high_amt_cnt')

    out = pd.concat([refund_grp, inst_rate, inst_max, div, ticket, hi_cnt], axis=1)
    return out

behavior_feats = build_behavior_features(full_tx)

# ============================================================
# 4. 시간/요일/시즌/파트·바이어 피처
# ============================================================
print("3) 시간·요일·시즌·파트 기반 피처 생성 중...")

def build_calendar_store_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data['sales_date'] = pd.to_datetime(data['sales_date'])
    data['sales_time'] = data['sales_time'].fillna(0).astype(int)

    # 방문일수
    visit_days = data.groupby('custid')['sales_date'].nunique().to_frame('cal_visit_days')

    def weekday_bucket(d):
        wd = d.dayofweek
        if wd <= 2: return 'WD_MonTueWed'
        elif 3 <= wd <= 4: return 'WD_ThuFri'
        else: return 'WD_Weekend'

    def season_bucket(d):
        m = d.month
        if 2 <= m <= 4: return 'SS_Spring'
        elif 5 <= m <= 7: return 'SS_Summer'
        elif 8 <= m <= 10: return 'SS_Fall'
        else: return 'SS_Winter'

    def time_bucket(t):
        if t < 1200: return 'TM_Morning'
        elif 1200 <= t < 1400: return 'TM_Lunch'
        elif 1400 <= t < 1600: return 'TM_Afternoon1'
        elif 1600 <= t < 1800: return 'TM_Afternoon2'
        else: return 'TM_Evening'

    data['cal_wd_group'] = data['sales_date'].apply(weekday_bucket)
    data['cal_season_group'] = data['sales_date'].apply(season_bucket)
    data['cal_time_group'] = data['sales_time'].apply(time_bucket)

    wd_pv = data.pivot_table(index='custid', columns='cal_wd_group',
                             values='tot_amt', aggfunc='size',
                             fill_value=0).add_prefix('cal_wd_')
    ss_pv = data.pivot_table(index='custid', columns='cal_season_group',
                             values='tot_amt', aggfunc='size',
                             fill_value=0).add_prefix('cal_season_')
    tm_pv = data.pivot_table(index='custid', columns='cal_time_group',
                             values='tot_amt', aggfunc='size',
                             fill_value=0).add_prefix('cal_time_')

    # 파트 정규화
    data['part_nm'] = data['part_nm'].fillna('').astype(str)
    part_norm = data['part_nm'].copy()
    replace_map = {
        '가정용품파트': '가정용품',
        '공산품파트': '공산품',
        '생식품파트': '생식품',
        '잡화파트': '잡화',
        '로얄부틱': '로얄부띠끄',
        '스포츠캐쥬얼': '스포츠캐주얼',
        '여성캐쥬얼': '여성캐주얼'
    }
    part_norm = part_norm.replace(replace_map)
    data['part_norm'] = part_norm

    male_parts = ['가정용품', '공산품', '생식품', '케주얼,구두,아동', '남성정장', '남성캐주얼']
    female_parts = ['여성캐주얼', '영캐릭터', '영플라자', '패션잡화', '여성정장']

    data['cal_male_part_flag'] = data['part_norm'].isin(male_parts).astype(int)
    data['cal_female_part_flag'] = data['part_norm'].isin(female_parts).astype(int)

    part_cnt = data.groupby('custid')[['cal_male_part_flag', 'cal_female_part_flag']].sum()
    part_cnt.columns = ['cal_male_part_cnt', 'cal_female_part_cnt']
    total = part_cnt['cal_male_part_cnt'] + part_cnt['cal_female_part_cnt']
    part_cnt['cal_male_ratio'] = part_cnt['cal_male_part_cnt'] / (total + 1e-5)
    part_cnt['cal_female_ratio'] = part_cnt['cal_female_part_cnt'] / (total + 1e-5)

    buyer_pv = data.pivot_table(index='custid', columns='buyer_nm',
                                values='tot_amt', aggfunc='size',
                                fill_value=0).add_prefix('cal_buyer_')

    out = pd.concat([visit_days, wd_pv, ss_pv, tm_pv, part_cnt, buyer_pv], axis=1)
    return out

calendar_feats = build_calendar_store_features(full_tx)

# ============================================================
# 5. 고객 관심 키워드 비중 피처
# ============================================================
print("4) 고객 관심 키워드 비중 피처 생성 중...")

def build_interest_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data['full_txt'] = (
        data['brd_nm'].fillna('') + ' ' +
        data['corner_nm'].fillna('') + ' ' +
        data['part_nm'].fillna('')
    )

    key_list = ['남성', '여성', '아동', '학생', '골프',
                '스포츠', '영', '마담', '캐주얼', '정장', '화장품']

    cust_total = data.groupby('custid')['tot_amt'].sum()
    idx = cust_total.index
    out = pd.DataFrame(index=idx)

    for kw in key_list:
        mask = data['full_txt'].str.contains(kw, na=False)
        amt_sum = data[mask].groupby('custid')['tot_amt'].sum()
        ratio = (amt_sum / (cust_total + 1e-5)).reindex(idx).fillna(0)
        out[f'int_ratio_{kw}'] = ratio

    return out

interest_ratio_feats = build_interest_ratio_features(full_tx)

# ============================================================
# 6. Entropy 피처 (브랜드 / 코너 다양성)
# ============================================================
print("5) Entropy 피처 생성 중...")

def build_entropy_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    def entropy_by_col(col):
        cnt = data.groupby(['custid', col]).size().unstack(fill_value=0)
        prob = cnt.div(cnt.sum(axis=1), axis=0)
        ent = prob.apply(lambda x: entropy(x), axis=1)
        return ent

    brd_ent = entropy_by_col('brd_nm').to_frame('ent_brd')
    corner_ent = entropy_by_col('corner_nm').to_frame('ent_corner')
    return pd.concat([brd_ent, corner_ent], axis=1)

entropy_feats = build_entropy_features(full_tx)

# ============================================================
# 7. Target Encoding (성별 목적 기반 인코딩)
# ============================================================
print("6) Target Encoding 피처 생성 중...")

full_tx['str_part_key'] = full_tx['str_nm'].astype(str) + '_' + full_tx['part_nm'].astype(str)

def kfold_target_encoding(trans_df, y_df, col_name, n_splits=5, noise=0.01):
    tx = trans_df.copy()
    if 'gender' not in tx.columns:
        tx = tx.merge(y_df, on='custid', how='left')

    scores = pd.DataFrame(index=tx['custid'].unique())
    new_col = f'te_{col_name}_gender'
    scores[new_col] = 0.0

    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    base_y = y_df[y_df['custid'].isin(tx['custid'])]

    for tr_idx, val_idx in kf.split(base_y, base_y['gender']):
        tr_ids = base_y.iloc[tr_idx]['custid']
        val_ids = base_y.iloc[val_idx]['custid']

        tr_tx = tx[tx['custid'].isin(tr_ids)].copy()
        tr_tx[col_name] = tr_tx[col_name].fillna('UNKNOWN')
        stats = tr_tx.groupby(col_name)['gender'].agg(['mean', 'count'])
        g_mean = tr_tx['gender'].mean()
        stats['te_value'] = np.where(stats['count'] < 10, g_mean, stats['mean'])

        val_tx = tx[tx['custid'].isin(val_ids)].copy()
        val_tx[col_name] = val_tx[col_name].fillna('UNKNOWN')
        val_tx = val_tx.merge(stats[['te_value']], on=col_name, how='left')
        val_tx['te_value'] = val_tx['te_value'].fillna(g_mean)
        val_tx['te_value'] += np.random.normal(0, noise, size=len(val_tx))

        cust_mean = val_tx.groupby('custid')['te_value'].mean()
        scores.loc[cust_mean.index, new_col] = cust_mean

    return scores

def target_encoding_for_test(train_trans, test_trans, y_df, col_name):
    tx = train_trans.merge(y_df, on='custid', how='left')
    tx[col_name] = tx[col_name].fillna('UNKNOWN')
    stats = tx.groupby(col_name)['gender'].agg(['mean', 'count'])
    g_mean = tx['gender'].mean()
    stats['te_value'] = np.where(stats['count'] < 10, g_mean, stats['mean'])

    test = test_trans.copy()
    test[col_name] = test[col_name].fillna('UNKNOWN')
    test = test.merge(stats[['te_value']], on=col_name, how='left')
    test['te_value'] = test['te_value'].fillna(g_mean)
    cust_mean = test.groupby('custid')['te_value'].mean()
    return cust_mean.to_frame(f'te_{col_name}_gender')

train_only = full_tx[full_tx['dataset'] == 'train'].copy()
test_only  = full_tx[full_tx['dataset'] == 'test'].copy()

te_blocks = []
for col in ['brd_nm', 'corner_nm', 'part_nm', 'str_part_key']:
    tr_te = kfold_target_encoding(train_only, y_train, col)
    te_te = target_encoding_for_test(train_only, test_only, y_train, col)
    te_block = pd.concat([tr_te, te_te], axis=0)
    te_blocks.append(te_block)

te_feats = pd.concat(te_blocks, axis=1)

# ============================================================
# 8. 기본 숫자/RFM 계열 + 점포/주말 비율
# ============================================================
print("7) 기본 수치/RFM 피처 생성 중...")

def build_base_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data['sales_date'] = pd.to_datetime(data['sales_date'])
    data['sales_time'] = data['sales_time'].fillna(0).astype(int).astype(str).str.zfill(4)
    data['base_hour'] = data['sales_time'].str[:2].astype(int)

    ref_date = data['sales_date'].max()
    recency = data.groupby('custid')['sales_date'].apply(
        lambda x: (ref_date - x.max()).days
    ).to_frame('base_recency')

    agg_dict = {
        'tot_amt': ['sum', 'mean', 'max', 'min', 'std'],
        'dis_amt': ['sum', 'mean'],
        'inst_mon': ['mean', 'max'],
        'sales_date': ['count'],
        'base_hour': ['mean', 'std']
    }
    agg = data.groupby('custid').agg(agg_dict)
    agg.columns = ['base_' + '_'.join(col).strip() for col in agg.columns]
    agg = agg.rename(columns={'base_sales_date_count': 'base_freq'})

    agg['base_ticket_avg'] = agg['base_tot_amt_sum'] / (agg['base_freq'] + 1e-5)
    agg['base_disc_rate_sum'] = agg['base_dis_amt_sum'] / (agg['base_tot_amt_sum'] + 1e-5)
    agg['base_disc_rate_mean'] = (agg['base_dis_amt_mean'] /
                                  (agg['base_tot_amt_mean'] + 1e-5)) * 100

    store_pv = data.pivot_table(index='custid', columns='str_nm',
                                values='tot_amt', aggfunc='size',
                                fill_value=0).add_prefix('base_str_cnt_')

    data['base_weekend_flag'] = data['sales_date'].dt.dayofweek >= 5
    weekend_ratio = data.groupby('custid')['base_weekend_flag'].mean().to_frame('base_weekend_ratio')

    out = pd.concat([agg, recency, store_pv, weekend_ratio], axis=1).fillna(0)
    return out

base_feats = build_base_features(full_tx)

# ============================================================
# 9. TF-IDF + SVD (브랜드/코너)
# ============================================================
print("8) TF-IDF + SVD 피처 생성 중...")

full_tx['brd_nm'] = full_tx['brd_nm'].astype(str).fillna('UNKNOWN')
full_tx['corner_nm'] = full_tx['corner_nm'].astype(str).fillna('UNKNOWN')

brd_text = full_tx.groupby('custid')['brd_nm'].apply(lambda x: ' '.join(x))
corner_text = full_tx.groupby('custid')['corner_nm'].apply(lambda x: ' '.join(x))

tfidf_brd = TfidfVectorizer(max_features=1000)
tfidf_corner = TfidfVectorizer(max_features=500)

brd_mat = tfidf_brd.fit_transform(brd_text)
corner_mat = tfidf_corner.fit_transform(corner_text)

svd_brd = TruncatedSVD(n_components=10, random_state=42)
svd_corner = TruncatedSVD(n_components=10, random_state=42)

brd_svd_df = pd.DataFrame(svd_brd.fit_transform(brd_mat),
                          index=brd_text.index).add_prefix('svd_brd_')
corner_svd_df = pd.DataFrame(svd_corner.fit_transform(corner_mat),
                             index=corner_text.index).add_prefix('svd_corner_')

tfidf_brd_small = TfidfVectorizer(max_features=200).fit_transform(brd_text)
tfidf_corner_small = TfidfVectorizer(max_features=100).fit_transform(corner_text)

brd_tfidf_df = pd.DataFrame(tfidf_brd_small.toarray(),
                            index=brd_text.index).add_prefix('tfidf_brd_')
corner_tfidf_df = pd.DataFrame(tfidf_corner_small.toarray(),
                               index=corner_text.index).add_prefix('tfidf_corner_')

# ============================================================
# 10. 할인 & 피크시간대 심화 피처
# ============================================================
print("9) 할인 + 피크시간대 피처 생성 중...")

def build_discount_peak_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data['sales_date'] = pd.to_datetime(data['sales_date'])
    data['sales_time'] = data['sales_time'].fillna(0).astype(int)
    data['dp_hour'] = data['sales_time'] // 100

    data['dp_disc_rate'] = np.where(
        data['tot_amt'] > 0,
        (data['dis_amt'] / (data['tot_amt'] + 1e-5)) * 100,
        0
    )
    data['dp_high_disc_flag'] = (data['dp_disc_rate'] >= 10).astype(int)
    data['dp_disc_flag'] = (data['dis_amt'] > 0).astype(int)

    disc_agg = data.groupby('custid').agg({
        'dp_disc_rate': ['mean', 'max'],
        'dp_high_disc_flag': ['sum', 'mean'],
        'dp_disc_flag': ['mean']
    })
    disc_agg.columns = [
        'dp_disc_mean', 'dp_disc_max',
        'dp_high_disc_cnt', 'dp_high_disc_ratio',
        'dp_disc_ratio'
    ]

    data['dp_lunch_flag'] = data['dp_hour'].between(11, 13).astype(int)
    data['dp_evening_flag'] = data['dp_hour'].between(17, 19).astype(int)

    peak_agg = data.groupby('custid').agg({
        'dp_lunch_flag': ['sum', 'mean'],
        'dp_evening_flag': ['sum', 'mean'],
        'dp_hour': ['std']
    })
    peak_agg.columns = [
        'dp_lunch_cnt', 'dp_lunch_ratio',
        'dp_evening_cnt', 'dp_evening_ratio',
        'dp_hour_std'
    ]

    out = pd.concat([disc_agg, peak_agg], axis=1)
    return out

discount_peak_feats = build_discount_peak_features(full_tx)

# ============================================================
# 11. 환불 강도/양상 피처
# ============================================================
print("10) 환불 강도/양상 피처 생성 중...")

def build_refund_intensity_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    if 'net_amt' in data.columns:
        base_amt_all = np.where(
            data['net_amt'].fillna(0) != 0,
            data['net_amt'].fillna(0),
            data['tot_amt'].fillna(0)
        )
    else:
        base_amt_all = data['tot_amt'].fillna(0).values

    data['ri_refund_flag'] = (base_amt_all < 0).astype(int)

    total_cnt = data.groupby('custid')['ri_refund_flag'].size().to_frame('ri_total_trx_cnt')

    ref_tx = data[data['ri_refund_flag'] == 1].copy()
    if ref_tx.empty:
        out = total_cnt.copy()
        out['ri_refund_cnt'] = 0
        out['ri_refund_ratio'] = 0.0
        out['ri_total_refund_amt'] = 0.0
        out['ri_mean_refund_amt'] = 0.0
        out['ri_max_refund_amt'] = 0.0
        out['ri_heavy_refund_flag'] = 0
        return out

    if 'net_amt' in ref_tx.columns:
        base_amt_ref = np.where(
            ref_tx['net_amt'].fillna(0) != 0,
            ref_tx['net_amt'].fillna(0),
            ref_tx['tot_amt'].fillna(0)
        )
    else:
        base_amt_ref = ref_tx['tot_amt'].fillna(0).values

    ref_tx['ri_refund_amt'] = np.abs(base_amt_ref)

    ref_amt_agg = ref_tx.groupby('custid')['ri_refund_amt'].agg(
        ['sum', 'mean', 'max', 'count']
    )
    ref_amt_agg.columns = [
        'ri_total_refund_amt', 'ri_mean_refund_amt',
        'ri_max_refund_amt', 'ri_refund_cnt'
    ]

    out = total_cnt.join(ref_amt_agg, how='left').fillna(0)

    out['ri_refund_ratio'] = out['ri_refund_cnt'] / (out['ri_total_trx_cnt'] + 1e-5)

    out['ri_heavy_refund_flag'] = (
        (out['ri_refund_ratio'] >= 0.10) |
        (out['ri_refund_cnt'] >= 3)
    ).astype(int)

    return out

refund_intensity_feats = build_refund_intensity_features(full_tx)

# ============================================================
# 12. Social Time 피처
# ============================================================
print("11) Social Time 피처 생성 중...")

def build_social_time_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data['sales_date'] = pd.to_datetime(data['sales_date'])
    data['sales_time'] = data['sales_time'].fillna(0).astype(int)

    data['st_hour'] = data['sales_time'] // 100
    dow = data['sales_date'].dt.dayofweek  # 월=0, ... 일=6

    is_weekday = dow <= 4
    is_weekend = dow >= 5

    social_flag = (
        (is_weekday & (data['st_hour'] >= 18)) |
        (is_weekend & (data['st_hour'].between(12, 21)))
    )

    data['st_social_flag'] = social_flag.astype(int)
    data['st_social_amt'] = data['tot_amt'] * data['st_social_flag']

    grp = data.groupby('custid')

    total_trx = grp['st_social_flag'].size().rename('st_total_trx_cnt')
    social_trx = grp['st_social_flag'].sum().rename('st_social_trx_cnt')
    social_trx_ratio = (social_trx / (total_trx + 1e-5)).rename('st_social_trx_ratio')

    total_amt = grp['tot_amt'].sum().rename('st_total_amt')
    social_amt = grp['st_social_amt'].sum().rename('st_social_amt')
    social_amt_ratio = (social_amt / (total_amt + 1e-5)).rename('st_social_amt_ratio')

    out = pd.concat(
        [total_trx, social_trx, social_trx_ratio,
         total_amt, social_amt, social_amt_ratio],
        axis=1
    )
    return out

social_time_feats = build_social_time_features(full_tx)

# ============================================================
# 13. 슬림 체류 시간(dwell) 피처
# ============================================================
print("12) 슬림 체류 시간(dwell) 피처 생성 중...")

def build_dwell_time_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data['sales_date'] = pd.to_datetime(data['sales_date'])
    data['sales_time'] = data['sales_time'].fillna(0).astype(int)

    # 시간(분) 변환
    hour = data['sales_time'] // 100
    minute = data['sales_time'] % 100
    minute = minute.clip(lower=0, upper=59)
    time_min = hour * 60 + minute
    data['dw_time_min'] = time_min

    # Social 트랜잭션 플래그
    dow = data['sales_date'].dt.dayofweek  # 월=0, ..., 일=6
    is_weekday = dow <= 4
    is_weekend = dow >= 5
    is_social_trx = (
        (is_weekday & (hour >= 18)) |
        (is_weekend & (hour.between(12, 21)))
    )
    data['dw_is_social_trx'] = is_social_trx.astype(int)

    # 하루 단위 그룹
    grp = data.groupby(['custid', 'sales_date'])

    # 하루별 첫/마지막 시간, 거래 수, 총 금액
    basic = grp['dw_time_min'].agg(['min', 'max', 'count']).rename(columns={'count': 'day_trx_cnt'})
    basic['span_min'] = (basic['max'] - basic['min']).clip(lower=0)

    day_amt = grp['tot_amt'].sum().rename('day_total_amt')

    # 하루 Social day 여부
    day_social_flag = grp['dw_is_social_trx'].max().rename('day_social_flag')

    day_df = pd.concat([basic, day_amt, day_social_flag], axis=1)

    # Longstay 플래그 (>= 60분)
    day_df['long_flag'] = (day_df['span_min'] >= 60).astype(int)

    # 분당 소비
    day_df['amt_per_dwell'] = day_df['day_total_amt'] / (day_df['span_min'] + 1)

    # custid 단위 집계
    cgrp = day_df.groupby('custid')
    span_stats = cgrp['span_min'].agg(['mean', 'max'])
    span_stats.columns = ['dw_span_mean_min', 'dw_span_max_min']

    longstay_ratio = cgrp['long_flag'].mean().rename('dw_longstay_ratio')
    amt_per_dwell_mean = cgrp['amt_per_dwell'].mean().rename('dw_amt_per_dwell_mean')

    # social_longstay_ratio: social day & long_flag=1 비율
    social_long_mask = (day_df['day_social_flag'] == 1) & (day_df['long_flag'] == 1)
    social_long_ratio = social_long_mask.groupby('custid').mean().rename('dw_social_longstay_ratio')

    out = pd.concat([
        span_stats,
        longstay_ratio,
        amt_per_dwell_mean,
        social_long_ratio
    ], axis=1).fillna(0)

    return out

dwell_feats = build_dwell_time_features(full_tx)

# ============================================================
# 14. 최근 K회(3,5) 행동 피처
# ============================================================
print("13) 최근 K회 행동 피처 생성 중...")

def build_recent_k_features(df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    data = df.copy()
    data['sales_dt'] = pd.to_datetime(data['sales_date'])
    data['sales_time'] = data['sales_time'].fillna(0).astype(int)
    hour = data['sales_time'] // 100
    minute = data['sales_time'] % 100
    minute = minute.clip(0, 59)
    data['sales_dt_full'] = data['sales_dt'] + pd.to_timedelta(hour, unit='h') + pd.to_timedelta(minute, unit='m')

    # 최신 거래부터 정렬
    data = data.sort_values(['custid', 'sales_dt_full'], ascending=[True, False])
    data['rn'] = data.groupby('custid').cumcount()

    recent = data[data['rn'] < k].copy()

    # 파트 정규화 (calendar와 동일 맵 사용)
    recent['part_nm'] = recent['part_nm'].fillna('').astype(str)
    replace_map = {
        '가정용품파트': '가정용품',
        '공산품파트': '공산품',
        '생식품파트': '생식품',
        '잡화파트': '잡화',
        '로얄부틱': '로얄부띠끄',
        '스포츠캐쥬얼': '스포츠캐주얼',
        '여성캐쥬얼': '여성캐주얼'
    }
    recent['part_norm'] = recent['part_nm'].replace(replace_map)

    male_parts = ['가정용품', '공산품', '생식품', '케주얼,구두,아동', '남성정장', '남성캐주얼']
    female_parts = ['여성캐주얼', '영캐릭터', '영플라자', '패션잡화', '여성정장']

    recent['recent_male_flag'] = recent['part_norm'].isin(male_parts).astype(int)
    recent['recent_female_flag'] = recent['part_norm'].isin(female_parts).astype(int)

    grp = recent.groupby('custid')

    out = pd.DataFrame(index=grp.size().index)
    out[f'recent_{k}_tot_amt_mean'] = grp['tot_amt'].mean()
    out[f'recent_{k}_tot_amt_max'] = grp['tot_amt'].max()
    out[f'recent_{k}_disc_rate_mean'] = (grp['dis_amt'].sum() / (grp['tot_amt'].sum() + 1e-5))
    out[f'recent_{k}_male_ratio'] = grp['recent_male_flag'].mean()
    out[f'recent_{k}_female_ratio'] = grp['recent_female_flag'].mean()

    return out.fillna(0)

recent5_feats = build_recent_k_features(full_tx, k=5)
recent3_feats = build_recent_k_features(full_tx, k=3)

# ============================================================
# 15. 모든 피처 통합
# ============================================================
print("14) 모든 피처 통합 중...")

all_features = base_feats \
    .join(style_feats, how='left') \
    .join(behavior_feats, how='left') \
    .join(calendar_feats, how='left') \
    .join(interest_ratio_feats, how='left') \
    .join(entropy_feats, how='left') \
    .join(brd_tfidf_df, how='left') \
    .join(corner_tfidf_df, how='left') \
    .join(brd_svd_df, how='left') \
    .join(corner_svd_df, how='left') \
    .join(te_feats, how='left') \
    .join(discount_peak_feats, how='left') \
    .join(refund_intensity_feats, how='left') \
    .join(dwell_feats, how='left') \
    .join(social_time_feats, how='left') \
    .join(recent5_feats, how='left') \
    .join(recent3_feats, how='left')

all_features.index.name = 'custid'
all_features = all_features.fillna(0)

# 컬럼명 정리 (특수문자 제거)
all_features.columns = [
    re.sub(r'[^0-9a-zA-Z가-힣_]', '_', str(c))
    for c in all_features.columns
]

print(f"✅ 통합 직후 피처 수: {all_features.shape[1]}")

# ============================================================
# 16. 추가 파생: pivot 비율 / gender score
# ============================================================
print("15) 비율/성향 스코어 파생 피처 생성 중...")

# 방문 수 (base_freq)를 이용한 비율
visit_cnt = all_features['base_freq'] + 1e-5

ratio_cols_prefixes = [
    'cal_wd_', 'cal_season_', 'cal_time_', 'cal_buyer_', 'base_str_cnt_'
]

for c in all_features.columns:
    if any(c.startswith(p) for p in ratio_cols_prefixes):
        all_features[c + '_ratio'] = all_features[c] / visit_cnt

# gender_part_score
if {'cal_male_ratio', 'cal_female_ratio'}.issubset(all_features.columns):
    all_features['gender_part_score'] = (
        all_features['cal_male_ratio'] - all_features['cal_female_ratio']
    )

# gender_interest_score (남성-여성 + 아동*0.5)
all_features['gender_interest_score'] = (
    all_features.get('int_ratio_남성', 0) -
    all_features.get('int_ratio_여성', 0) +
    0.5 * all_features.get('int_ratio_아동', 0)
)

print(f"➕ 파생 이후 피처 수: {all_features.shape[1]}")

# ============================================================
# 17. log1p + 이상치 클리핑 (금액/횟수 계열)
# ============================================================
print("16) log1p + 클리핑 전처리 중...")

num_cols = [
    c for c in all_features.columns
    if all_features[c].dtype.kind in ['i', 'u', 'f', 'b']
]

for c in num_cols:
    # 금액/합/횟수/거리/span/recency 계열만 대상으로 (너무 공격적 방지)
    if any(kw in c for kw in ['amt', 'sum', 'ticket', 'refund', 'recency', 'span', 'freq', 'cnt', 'total']):
        col = all_features[c].astype(float)

        # 음수는 0으로 올림 (log 안정성)
        col = np.where(col < 0, 0, col)

        q_low, q_high = np.nanpercentile(col, [1, 99])
        col = np.clip(col, q_low, q_high)

        all_features[c] = np.log1p(col)

print("🔁 log1p + 클리핑 완료")

# ============================================================
# 18. Low-variance / Too-sparse 컬럼 제거
# ============================================================
print("17) Low-variance / Sparse 피처 제거 중...")

var = all_features.var()
low_var_cols = var[var < LOW_VAR_TH].index.tolist()
print(f"📉 Low-variance cols 제거: {len(low_var_cols)}개")
all_features = all_features.drop(columns=low_var_cols)

nonzero_ratio = (all_features != 0).mean(axis=0)
sparse_cols = nonzero_ratio[nonzero_ratio < SPARSE_TH].index.tolist()
print(f"📉 Too-sparse cols 제거: {len(sparse_cols)}개 (threshold={SPARSE_TH})")
all_features = all_features.drop(columns=sparse_cols)

print(f"✅ 정리 후 최종 피처 수: {all_features.shape[1]}")

# ============================================================
# 19. Train / Test 분리
# ============================================================
X_train = all_features.loc[y_train['custid']]
X_test  = all_features.loc[test_tx['custid'].unique()]
y = y_train['gender']

print("Train shape:", X_train.shape)
print("Test  shape:", X_test.shape)

# ============================================================
# 20. CatBoost 단일 모델 (seed=42 고정)
# ============================================================
best_params_catboost = {
    'iterations': 1658,
    'learning_rate': 0.0107,
    'depth': 7,
    'l2_leaf_reg': 13.9,
    'subsample': 0.708,
    'colsample_bylevel': 0.818,
    'min_data_in_leaf': 39,
    'random_strength': 3.15,
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 0,
    'allow_writing_files': False
}

print("\n🔥 CatBoost 단일 모델(seed=42) 학습 시작...")

model = CatBoostClassifier(**best_params_catboost)
model.fit(X_train, y)

test_pred = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'custid': X_test.index,
    'gender': test_pred
})

out_name = 'submission_catboost_single_seed42_enhanced.csv'
submission.to_csv(out_name, index=False)
print(f"\n🎉 완료! '{out_name}' 저장됨.")

1) 쇼핑 스타일 피처 생성 중...
2) 행동 패턴 피처 생성 중...
3) 시간·요일·시즌·파트 기반 피처 생성 중...
4) 고객 관심 키워드 비중 피처 생성 중...
5) Entropy 피처 생성 중...
6) Target Encoding 피처 생성 중...
7) 기본 수치/RFM 피처 생성 중...
8) TF-IDF + SVD 피처 생성 중...
9) 할인 + 피크시간대 피처 생성 중...
10) 환불 강도/양상 피처 생성 중...
11) Social Time 피처 생성 중...
12) 슬림 체류 시간(dwell) 피처 생성 중...
13) 최근 K회 행동 피처 생성 중...
14) 모든 피처 통합 중...
✅ 통합 직후 피처 수: 461
15) 비율/성향 스코어 파생 피처 생성 중...
➕ 파생 이후 피처 수: 514
16) log1p + 클리핑 전처리 중...
🔁 log1p + 클리핑 완료
17) Low-variance / Sparse 피처 제거 중...
📉 Low-variance cols 제거: 5개
📉 Too-sparse cols 제거: 12개 (threshold=0.003)
✅ 정리 후 최종 피처 수: 497
Train shape: (30000, 497)
Test  shape: (19995, 497)

🔥 CatBoost 단일 모델(seed=42) 학습 시작...

🎉 완료! 'submission_catboost_single_seed42_enhanced.csv' 저장됨.


In [16]:
# ============================================================
# Train & Test 저장
# ============================================================
X_train.reset_index().to_csv('X_train_1st.csv', encoding='cp949', index=False)
X_test.reset_index().to_csv('X_test_1st.csv', encoding='cp949', index=False)

In [5]:
# ============================================================
# [전체 방법론 개요]
# ------------------------------------------------------------
# - 목표: 개별 거래 로그(train_transactions, test_transactions)를
#   고객 단위(custid) 피처로 집계한 뒤, CatBoost 단일 모델로 성별(gender)을 예측한다.
#
# - 전처리/구조:
#   1) train/test 거래 데이터를 통합(full_tx) 후 공통 스키마로 맞춘다.
#   2) 모든 피처는 최종적으로 고객 단위(custid index)의 wide table(all_features)로 집계한다.
#
# - 주요 피처 그룹(섹션 번호 기준):
#   (2) 쇼핑 스타일: corner_nm 텍스트에서 '명품/수입/행사' 키워드를 추출해
#       명품 성향(st_ratio_luxury), 행사/알뜰 성향(st_ratio_bargain),
#       할인율의 평균·표준편차(st_disc_mean, st_disc_std)를 계산해
#       고객의 가격 민감도/명품 선호도를 표현.
#
#   (3) 행동 패턴: 환불 여부, 할부 사용 여부, 브랜드/코너/파트 다양성,
#       객단가, 고액 결제 횟수 등을 이용해 소비 성향(충동/고가/분산)을 표현.
#
#   (4) 시간·요일·시즌·파트/바이어: 판매 일자/시간을
#       요일/시즌/시간대 버킷으로 나누고(pivot),
#       남성/여성 중심 파트(part_nm 정규화 후 분류) 방문 비율,
#       바이어(buyer_nm)별 방문 패턴을 생성하여
#       "언제/어디서/어떤 파트"를 주로 이용하는지 캡처.
#
#   (5) 관심 키워드 비중: brd_nm, corner_nm, part_nm를 하나의 텍스트로 합쳐
#       '남성/여성/아동/스포츠/캐주얼/정장/화장품' 등의 키워드가
#       전체 결제 금액 대비 어느 정도 비중을 차지하는지 비율(int_ratio_*)로 표현.
#
#   (6) Entropy 피처: 브랜드/코너별 방문 분포의 샤논 엔트로피를 계산하여
#       특정 브랜드/코너에 몰려 있는지, 다양한 곳을 이용하는지(ent_brd, ent_corner) 측정.
#
#   (7) Target Encoding: brd_nm, corner_nm, part_nm, str_part_key 등의
#       고카디널리티 범주형 피처에 대해 K-Fold target encoding을 적용해
#       성별 조건부 평균을 안정적으로 인코딩.
#
#   (8) 기본 RFM/수치 피처: 구매 Recency, 구매 빈도(base_freq),
#       금액(sum/mean/max/min/std), 할인 금액/비율, 시간대 통계(base_hour),
#       점포별 방문 횟수(base_str_cnt_*), 주말 방문 비율 등을 계산해
#       전통적인 RFM + 점포 이용 패턴을 반영.
#
#   (9) TF-IDF + SVD: 고객별 브랜드/코너 시퀀스를 문서로 보고
#       TF-IDF 임베딩 후 SVD로 차원 축소(svd_brd_*, svd_corner_*)하여
#       고차원 카테고리 정보를 압축된 연속형 피처로 사용.
#       동시에 상위 일부 TF-IDF(raw) 피처(tfidf_brd_*, tfidf_corner_*)도 함께 사용해
#       세밀한 브랜드·코너 패턴을 반영.
#
#   (10) 할인·피크타임: 할인율/할인 여부, 점심/저녁 피크 시간대 방문 비율,
#        시간 분산(dp_hour_std) 등을 통해 "언제, 얼마나 할인받아 소비하는지"를 표현.
#
#   (11) 환불 강도: 환불 거래만 따로 떼어
#        환불 횟수/비율, 환불 금액 합·평균·최대, heavy_refund_flag 등을 만들고
#        '환불이 잦은 고객' 특성을 반영.
#
#   (12) Social Time: 평일 저녁·주말 낮~저녁 등 사회적 활동 시간이 포함된
#        거래 비율/금액 비율(st_social_trx_ratio, st_social_amt_ratio)을 계산해
#        "사회적 시간대" 소비 패턴을 캡처.
#
#   (13) Dwell Time(체류 시간 근사): 하루 단위로 첫 거래~마지막 거래 시간 차이를
#        분 단위로 계산(span_min), 60분 이상 체류 longstay 비율,
#        체류 시간당 금액, social day에서의 longstay 비율 등을 통해
#        체류 시간 기반 오프라인 이용 강도를 표현.
#
#   (14) 최근 K회(3/5) 행동: 고객별 최신 거래 3건/5건만 따로 떼어
#        금액 평균/최대, 할인율, 남성/여성 파트 비율 등을 계산하여
#        "최근" 성향이 과거 평균과 다를 경우도 반영.
#
#   (16) 추가 파생: 요일/시즌/시간/점포/바이어 피벗 카운트를
#        방문 수(base_freq)로 나눈 비율(ratio) 피처를 생성하고,
#        남성/여성 파트/키워드 비율을 결합하여 gender_part_score,
#        gender_interest_score 등 성별 성향 스코어를 구성.
#
#   (17) 수치 안정화: 금액/횟수 계열 피처에 대해
#        1~99퍼센타일 범위로 클리핑 + log1p 변환을 적용해
#        극단값을 완화하고 CatBoost가 학습하기 좋은 스케일로 맞춤.
#
#   (18) 피처 선택: 매우 분산이 작은(low-variance) 피처와
#        거의 대부분 0인(sparse) 피처를 제거해 노이즈를 줄이고
#        모델 복잡도를 제어.
#
# - 모델링:
#   • 전처리된 all_features를 기준으로 train/test를 custid로 매핑.
#   • CatBoostClassifier 단일 모델(앙상블 X) 사용, 평가 지표는 AUC.
#   • 사전에 튜닝된 하이퍼파라미터(best_params_catboost, seed=42 고정)를 이용해
#     전체 train 데이터에 학습 후 test에 대한 성별 확률을 예측하여 제출.
# ============================================================
